# Test Vertex AI Answer Method with Sessions

This notebook tests the answer method implementation for multi-turn conversations.
"""
Test Vertex AI Answer Method with Sessions

Run this script to test the answer method implementation.
Can also be copy-pasted into a Jupyter notebook.

PARAMETER ALIGNMENT WITH vertex_search.py:
==========================================

vertex_search.py (Search Method)          →  Answer Method Equivalent
-----------------------------------------    ----------------------------------------
summary_result_count=5                    →  search_spec.search_params.max_return_results=5
summary_include_citations=True            →  answer_generation_spec.include_citations=True
ignore_adversarial_query=True             →  answer_generation_spec.ignore_adversarial_query + query_classification
ignore_jail_breaking_query=True           →  answer_generation_spec.ignore_jail_breaking_query=True
ignore_low_relevant_content=True          →  answer_generation_spec.ignore_low_relevant_content=False (handled differently)
language_code="en"                        →  answer_generation_spec.answer_language_code="en"
summary_prompt=SUMMARY_PROMPT             →  answer_generation_spec.prompt_spec.preamble=PROMPT
get_extractive_answers=False              →  N/A (answer method doesn't use extractive segments)
max_extractive_segment_count=3            →  N/A
use_semantic_chunks=True                  →  Built-in to answer method

NEW FEATURES IN ANSWER METHOD:
==============================
- query_understanding_spec.query_rephraser_spec: Improves multi-turn understanding
- query_understanding_spec.query_classification_spec: Classify query types
- related_questions_spec: Generate follow-up questions for users
- session: Session-based conversation context (maps to LangGraph thread_id)
- user_pseudo_id: Track users for analytics

TESTS INCLUDED:
===============
1. Session creation and basic query
2. Follow-up query with session continuity (multi-turn)
3. Variable search result counts (5 vs 10)
4. Query classification for out-of-scope queries
"""

In [2]:


import os
import sys
from pathlib import Path

# Load environment
from dotenv import load_dotenv
load_dotenv('../.env')

# Imports
from google.cloud import discoveryengine_v1

# Configuration
PROJECT_ID = os.getenv('GOOGLE_CLOUD_PROJECT')
ENGINE_ID = os.getenv('VERTEX_AI_SEARCH_ENGINE_ID') or os.getenv('VERTEX_AI_SEARCH_DATA_STORE_ID')
LOCATION = 'global'
SERVING_CONFIG = 'default_serving_config'

print(f"Configuration:")
print(f"  Project: {PROJECT_ID}")
print(f"  Engine: {ENGINE_ID}")
print(f"  Location: {LOCATION}")
print()

# Create client
client = discoveryengine_v1.ConversationalSearchServiceClient()

# Build paths
serving_config_path = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/"
    f"collections/default_collection/engines/{ENGINE_ID}/"
    f"servingConfigs/{SERVING_CONFIG}"
)

print(f"Serving config: {serving_config_path}")
print()


Configuration:
  Project: nedo-ai-safety-citadel-ai
  Engine: test-periodic-pdfs_1760523041262
  Location: global

Serving config: projects/nedo-ai-safety-citadel-ai/locations/global/collections/default_collection/engines/test-periodic-pdfs_1760523041262/servingConfigs/default_serving_config



E0000 00:00:1760524631.732176 9635760 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [ ]:

# ============================================================================
# TEST 1: First Query - Create Session
# ============================================================================
print("="*80)
print("TEST 1: First Query - Create Session")
print("="*80)

session_wildcard = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/"
    f"collections/default_collection/engines/{ENGINE_ID}/"
    f"sessions/-"  # Wildcard for auto-creation
)

query_1 = "Where do I renew my healthcare insurance in Yokohama?"
print(f"Query: {query_1}")
print(f"Session: {session_wildcard} (auto-create)")
print()

request_1 = discoveryengine_v1.AnswerQueryRequest(
    serving_config=serving_config_path,
    query=discoveryengine_v1.Query(text=query_1),
    session=session_wildcard,
    
    # Query Understanding Spec - aligns with search method's query processing
    query_understanding_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec(
        query_rephraser_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryRephraserSpec(
            disable=False,  # Enable query rephrasing for better multi-turn understanding
            max_rephrase_steps=1,
        ),
        query_classification_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec(
            types=[
                discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec.Type.ADVERSARIAL_QUERY,
                discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec.Type.NON_ANSWER_SEEKING_QUERY,
            ]
        ),
    ),
    
    # Answer Generation Spec - aligns with search method's summary_spec
    answer_generation_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec(
        model_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec.ModelSpec(
            model_version="gemini-2.5-pro-001/answer_gen/v1",
        ),
        prompt_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec.PromptSpec(
            preamble="You are a helpful assistant for Japanese official procedures.",
        ),
        include_citations=True,
        
        # Safety filters - aligns with vertex_search.py safety settings
        ignore_adversarial_query=False,  # Let query classification handle it
        ignore_non_answer_seeking_query=False,  # Let query classification handle it
        ignore_jail_breaking_query=True,  # Same as search method
        ignore_low_relevant_content=False,  # Answer method handles this differently
        
        answer_language_code="en",
    ),
    
    # Search Spec - aligns with search method's result count and extractive settings
    search_spec=discoveryengine_v1.AnswerQueryRequest.SearchSpec(
        search_params=discoveryengine_v1.AnswerQueryRequest.SearchSpec.SearchParams(
            max_return_results=5,  # Aligns with summary_result_count=5
            # Note: extractive segments handled differently in answer method
        ),
    ),
    
    # Related Questions Spec - NEW: Generate follow-up questions
    related_questions_spec=discoveryengine_v1.AnswerQueryRequest.RelatedQuestionsSpec(
        enable=True,  # Generate related questions for user
    ),
    
    # User tracking
    user_pseudo_id="test-user-multiturn",
)

response_1 = client.answer_query(request_1)

print(response_1)

print("ANSWER:")
print("-"*80)
if hasattr(response_1, 'answer') and response_1.answer:
    print(response_1.answer.answer_text)
    print()
    print(f"Citations: {len(response_1.answer.citations)}")
    
    # Show citation details
    if response_1.answer.citations:
        print("\nCitation Details:")
        for idx, citation in enumerate(response_1.answer.citations, 1):
            print(f"  [{idx}] {citation.title if hasattr(citation, 'title') else 'N/A'}")
            if hasattr(citation, 'sources'):
                for source in citation.sources:
                    if hasattr(source, 'uri'):
                        print(f"      URI: {source.uri}")

# Extract session ID
session_id = None
if hasattr(response_1, 'session') and response_1.session:
    session_name = response_1.session.name
    session_id = session_name.split('/sessions/')[-1]
    print(f"\n✨ Created session: {session_id}")
else:
    print(f"\n⚠️ No session in response")

# Show query classification results
if hasattr(response_1, 'answer') and response_1.answer:
    answer = response_1.answer
    if hasattr(answer, 'answer_skipped_reasons') and answer.answer_skipped_reasons:
        print(f"\n⚠️ Answer skipped reasons: {answer.answer_skipped_reasons}")

print()


TEST 1: First Query - Create Session
Query: Where do I renew my healthcare insurance in Yokohama?
Session: projects/nedo-ai-safety-citadel-ai/locations/global/collections/default_collection/engines/test-periodic-pdfs_1760523041262/sessions/- (auto-create)

answer {
  name: "projects/634361342501/locations/global/collections/default_collection/engines/test-periodic-pdfs_1760523041262/sessions/15623789970899647039/answers/15623789970899648276"
  state: SUCCEEDED
  answer_text: "To renew your National Health Insurance (Kokuho) in Yokohama, you should go to the municipal government division in charge of National Health Insurance. You must notify them within 14 days when enrolling in or quitting National Health Insurance. A health insurance card is required when receiving medical care. If you lose or break your Health Insurance Card, apply for a card reissuance at the municipal government division in charge of National Health Insurance.\n"
  citations {
    end_index: 152
    sources {
    

In [5]:

# ============================================================================
# TEST 2: Follow-up Query - Test Session Continuity
# ============================================================================
if session_id:
    print("="*80)
    print("TEST 2: Follow-up Query - Test Session Continuity")
    print("="*80)
    
    session_path = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/"
        f"collections/default_collection/engines/{ENGINE_ID}/"
        f"sessions/{session_id}"
    )
    
    query_2 = "What documents do I need?"
    print(f"Query: {query_2}")
    print(f"Session: ...{session_id[-8:]} (reusing)")
    print()
    
    request_2 = discoveryengine_v1.AnswerQueryRequest(
        serving_config=serving_config_path,
        query=discoveryengine_v1.Query(text=query_2),
        session=session_path,
        
        query_understanding_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec(
            query_rephraser_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryRephraserSpec(
                disable=False,
                max_rephrase_steps=1,
            ),
        ),
        
        answer_generation_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec(
            model_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec.ModelSpec(
                model_version="gemini-2.0-flash-001/answer_gen/v1",
            ),
            prompt_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec.PromptSpec(
                preamble="You are a helpful assistant for Japanese official procedures.",
            ),
            include_citations=True,
            ignore_adversarial_query=False,
            ignore_non_answer_seeking_query=False,
            ignore_jail_breaking_query=True,
            ignore_low_relevant_content=False,
            answer_language_code="en",
        ),
        
        search_spec=discoveryengine_v1.AnswerQueryRequest.SearchSpec(
            search_params=discoveryengine_v1.AnswerQueryRequest.SearchSpec.SearchParams(
                max_return_results=5,
            ),
        ),
        
        related_questions_spec=discoveryengine_v1.AnswerQueryRequest.RelatedQuestionsSpec(
            enable=True,
        ),
        
        user_pseudo_id="test-user-multiturn",
    )
    
    response_2 = client.answer_query(request_2)
    
    print("ANSWER (Should understand context - healthcare insurance):")
    print("-"*80)
    if hasattr(response_2, 'answer') and response_2.answer:
        print(response_2.answer.answer_text[:500] + "..." if len(response_2.answer.answer_text) > 500 else response_2.answer.answer_text)
        print(f"\nCitations: {len(response_2.answer.citations)}")
    
    if hasattr(response_2, 'related_questions') and response_2.related_questions:
        print(f"\n🔗 Related Questions:")
        for q in response_2.related_questions:
            print(f"  - {q}")
    
    print()


TEST 2: Follow-up Query - Test Session Continuity
Query: What documents do I need?
Session: ...99647039 (reusing)

ANSWER (Should understand context - healthcare insurance):
--------------------------------------------------------------------------------
The provided documents do not list the documents needed to renew your National Health Insurance. However, the documents state that you must notify the staff in charge of National Health Insurance at your municipal government when enrolling in or quitting National Health Insurance. If you lose or break your Health Insurance Card, you can apply for a card reissuance at the municipal government division in charge of National Health Insurance.


Citations: 2



In [ ]:

# ============================================================================
# TEST 3: Test with More Search Results
# ============================================================================
print("="*80)
print("TEST 3: Test with More Search Results")
print("="*80)

query_3 = "What are the visa requirements for students in Japan?"
print(f"Query: {query_3}")
print(f"max_return_results: 10 (increased from 5)")
print()

request_3 = discoveryengine_v1.AnswerQueryRequest(
    serving_config=serving_config_path,
    query=discoveryengine_v1.Query(text=query_3),
    session=(
        f"projects/{PROJECT_ID}/locations/{LOCATION}/"
        f"collections/default_collection/engines/{ENGINE_ID}/"
        f"sessions/-"
    ),
    
    query_understanding_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec(
        query_rephraser_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryRephraserSpec(
            disable=False,
            max_rephrase_steps=1,
        ),
    ),
    
    answer_generation_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec(
        model_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec.ModelSpec(
            model_version="gemini-2.0-flash-001/answer_gen/v1",
        ),
        include_citations=True,
        answer_language_code="en",
    ),
    
    # Increased search results
    search_spec=discoveryengine_v1.AnswerQueryRequest.SearchSpec(
        search_params=discoveryengine_v1.AnswerQueryRequest.SearchSpec.SearchParams(
            max_return_results=10,  # More results for comprehensive answer
        ),
    ),
    
    related_questions_spec=discoveryengine_v1.AnswerQueryRequest.RelatedQuestionsSpec(
        enable=True,
    ),
    
    user_pseudo_id="test-user-multiturn",
)

response_3 = client.answer_query(request_3)

print("ANSWER:")
print("-"*80)
if hasattr(response_3, 'answer') and response_3.answer:
    print(response_3.answer.answer_text[:500] + "..." if len(response_3.answer.answer_text) > 500 else response_3.answer.answer_text)
    print(f"\nCitations: {len(response_3.answer.citations)}")

print()

# ============================================================================
# TEST 4: Test Query Classification (Out of Scope)
# ============================================================================
print("="*80)
print("TEST 4: Test Query Classification (Out of Scope)")
print("="*80)

query_4 = "What's the best ramen restaurant in Tokyo?"
print(f"Query: {query_4}")
print("Expected: Should be classified as non-answer-seeking or out of scope")
print()

request_4 = discoveryengine_v1.AnswerQueryRequest(
    serving_config=serving_config_path,
    query=discoveryengine_v1.Query(text=query_4),
    session=(
        f"projects/{PROJECT_ID}/locations/{LOCATION}/"
        f"collections/default_collection/engines/{ENGINE_ID}/"
        f"sessions/-"
    ),
    
    query_understanding_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec(
        query_classification_spec=discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec(
            types=[
                discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec.Type.ADVERSARIAL_QUERY,
                discoveryengine_v1.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec.Type.NON_ANSWER_SEEKING_QUERY,
            ]
        ),
    ),
    
    answer_generation_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec(
        model_spec=discoveryengine_v1.AnswerQueryRequest.AnswerGenerationSpec.ModelSpec(
            model_version="gemini-2.0-flash-001/answer_gen/v1",
        ),
        include_citations=True,
        ignore_non_answer_seeking_query=True,  # Should skip answer for this
        answer_language_code="en",
    ),
    
    search_spec=discoveryengine_v1.AnswerQueryRequest.SearchSpec(
        search_params=discoveryengine_v1.AnswerQueryRequest.SearchSpec.SearchParams(
            max_return_results=5,
        ),
    ),
    
    user_pseudo_id="test-user-multiturn",
)

response_4 = client.answer_query(request_4)

print("RESULT:")
print("-"*80)
if hasattr(response_4, 'answer') and response_4.answer:
    if response_4.answer.answer_text:
        print(f"Answer provided: {response_4.answer.answer_text[:200]}...")
    else:
        print("No answer text")
    
    if hasattr(response_4.answer, 'answer_skipped_reasons') and response_4.answer.answer_skipped_reasons:
        print(f"\n✅ Answer skipped reasons: {response_4.answer.answer_skipped_reasons}")
    else:
        print("\nNo skip reasons (answer was generated)")

print()

# ============================================================================
# SUMMARY
# ============================================================================
print("="*80)
print("SUMMARY")
print("="*80)
print("✅ Test 1: Session creation and basic query")
if session_id:
    print(f"✅ Session ID: {session_id}")
    print("✅ Test 2: Follow-up query with session continuity")
print("✅ Test 3: Increased search results (10 vs 5)")
print("✅ Test 4: Query classification testing")
print()
print("Key Features Tested:")
print("- ✅ Session creation and persistence")
print("- ✅ Query understanding and rephrasing")
print("- ✅ Query classification (adversarial, non-answer-seeking)")
print("- ✅ Related questions generation")
print("- ✅ Citation extraction")
print("- ✅ Safety filters (jail breaking, low relevance)")
print("- ✅ Variable search result counts")
print()
print("Parameters aligned with vertex_search.py:")
print("- max_return_results (aligned with summary_result_count)")
print("- Safety filters (ignore_jail_breaking_query, etc.)")
print("- Language code")
print("- Citation inclusion")
print()
print("New features in answer method:")
print("- Related questions generation")
print("- Query rephrasing for multi-turn")
print("- Query classification")
print("- Session-based context persistence")
